# Catalogs & Plate solving

It is often required to match your detected stars with a catalog. Let's load an example image (from an archive like SDSS) and see how to make queries from a catalog

In [1]:
from prose.archive import sdss_image

# an image of TRAPPIST-1
image = sdss_image(("23 06 29.3684", "-05 02 29.0373"), (20, 20))
image.show()


In our case, the image is plate solved, we can check with

In [2]:
image.plate_solved

## Querying a catalog

To query a catalog we can use a catalog block from the `prose.blocks.catalogs` module

In [3]:
from prose.blocks import catalogs, detection

# Detect sources before querying the catalog
image = detection.PointSourceDetection()(image)

image = catalogs.GaiaCatalog(mode="replace")(image)

# visualizing the catalog stars
image.show()

An overlay with labels can be plotted with

In [4]:
# show cutout around 13-th source
cutout = image.cutout(13, 300, reset_index=False)
cutout.show()
cutout.plot_catalog("gaia", label=True, color="w")

We see here that the `Image.sources` (plotted by default with `image.show`) are set to the queried stars

```{note}
If instead you want to cross-match the queried stars to already existing ``Image.sources``, use ``catalogs.GaiaCatalog(mode='crossmatch')``. This way the index of the ``Image.sources`` will be the same as the index in the catalog (see the dataframe below)
```

The full catalogs can be found at

In [5]:
image.catalogs["gaia"]

## Plate solving

To plate solve an image we can use the following sequence

In [6]:
from prose import Sequence, blocks

plate = Sequence(
    [
        blocks.detection.PointSourceDetection(n=30),
        blocks.catalogs.PlateSolve(debug=True),
    ]
)

plate.run(image, show_progress=False)


Seeing the markers on the stars (only with `debug=True`) in the image shows that the plate solving was successful

```{note}
``PlateSolve`` is slow so it is not recommended to use this block in a sequence with more than 5 images. Instead you can pass a plate-solved ``ref_image`` to this block so that catalog stars are queried only once
```